In [1]:
import pandas as pd
import string
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
nlp = spacy.load("en_core_web_sm", exclude=["parser", "ner"])

In [3]:
print("Loading dataset...")

try:
    df = pd.read_json('C:\\Users\\andre\\Desktop\\Magistrale\\TLN\\Esercitazione8_word2VecMulti\\archive\\News_Category_Dataset_v3.json', lines=True)
except FileNotFoundError:
    print("Error: Please download the dataset from Kaggle and place it in the directory.")
    exit()

Loading dataset...


In [4]:
# Randomly select a subset of 10,000 headlines
subset_df = df.sample(n=10000, random_state=42).reset_index(drop=True)
headlines = subset_df['headline'].tolist()

def preprocess_query(text):
    """
    Tokenizes, lowercases, removes stopwords/punctuation, and lemmatizes a single string.
    Usato per la query dell'utente.
    """
    doc = nlp(text)
    processed_tokens = [
        str(token.lemma_).lower() 
        for token in doc 
        if not token.is_stop and not token.is_punct
    ]
    return " ".join(processed_tokens)

In [5]:
print("Preprocessing text in batch")
preprocessed_docs = []

# Sfruttiamo nlp.pipe per la massima efficienza sui 10.000 documenti
for doc in nlp.pipe(headlines, batch_size=256):
    processed_tokens = [
        str(token.lemma_).lower() 
        for token in doc 
        if not token.is_stop and not token.is_punct
    ]
    preprocessed_docs.append(" ".join(processed_tokens))

#Vector Space Model Representation
print("Generating TF-IDF vectors...")
vectorizer = TfidfVectorizer()

# Convert preprocessed documents to TF-IDF matrix
tfidf_matrix = vectorizer.fit_transform(preprocessed_docs)

# The vocabulary is stored inside the vectorizer
print(f"Vocabulary size: {len(vectorizer.vocabulary_)} words")

Preprocessing text in batch
Generating TF-IDF vectors...
Vocabulary size: 13598 words


In [6]:
# Document Retrieval
def retrieve_documents(query, vectorizer, tfidf_matrix, original_docs, top_n=5):
    # Preprocess the query using the spacy function
    preprocessed_query = preprocess_query(query)
    
    # Convert the query to a TF-IDF vector
    query_vector = vectorizer.transform([preprocessed_query])
    
    # Compute cosine similarity between query and all documents
    similarities = cosine_similarity(query_vector, tfidf_matrix).flatten()
    
    # Get the indices of the top N most similar documents
    top_indices = similarities.argsort()[-top_n:][::-1]
    
    print(f"\n--- Top {top_n} Results for: '{query}' ---")
    for i, idx in enumerate(top_indices):
        score = similarities[idx]
        print(f"{i+1}. [Score: {score:.4f}] {original_docs[idx]}")
        
    return top_indices

if __name__ == "__main__":
    print("\n--- Document Retrieval System Ready ---")
    while True:
        user_query = input("\nEnter a query (or type 'exit' to quit): ")
        if user_query.lower() == 'exit':
            break
        if not user_query.strip():
            continue
            
        retrieve_documents(user_query, vectorizer, tfidf_matrix, headlines)


--- Document Retrieval System Ready ---

--- Top 5 Results for: 'government budget deficit' ---
1. [Score: 0.3392] On a Budget? 5 Ways You Can Still Travel the World!
2. [Score: 0.2925] When It Comes To Corporate Tax Breaks, Congress Stops Caring About The Deficit
3. [Score: 0.2883] Government Of, By, And For The Plutocrats
4. [Score: 0.2852] A Paul Ryan Budget In A Donald Trump World
5. [Score: 0.2685] 2nd-Grader Expertly Schools Donald Trump On How Trade Deficits Work

--- Top 5 Results for: 'basketball ' ---
1. [Score: 0.4905] Spider-Man Plays Pick-Up Basketball With Kids
2. [Score: 0.4104] A High School Cancels The Basketball Season Amid Rape Charges
3. [Score: 0.4025] College Basketball Player Dribbles Out Clock While His Team Is Losing By 1
4. [Score: 0.3989] CNN Analyst Compares Syria Strike To Kentucky Basketball
5. [Score: 0.3914] David Stern, NBA Commissioner, Says Obama's 'Not That Good' At Basketball (VIDEO)

--- Top 5 Results for: 'stock market crash ' ---
1. [Score: 0.55

# Query di esempio:
- "government budget deficit" (Politics) 
- "apple smartphone release" (Tech)
- "basketball championship finals" (Sports)
- "climate change emissions" (Environment)
- "oscar awards red carpet" (Entertainment)
- "stock market crash" (Business)
- "healthy diet recipes" (Wellness/Food)
- "supreme court ruling" (Politics/U.S. News)
- "artificial intelligence breakthrough" (Tech/Science)
- "travel restrictions europe" (Travel)



Lexical Gap: Because TF-IDF is a sparse embedding technique based purely on word matching, if a query uses the word "smartphone" but the headline says "iPhone", the cosine similarity might be 0. This illustrates the limitation of VSM compared to dense embeddings (like Word2Vec or BERT) which capture semantic meaning.

Sparsity: Short headlines may not contain enough overlapping words with your query, potentially leading to low similarity scores across the board. If a headline is only 4 words long, removing stopwords might leave only 1 or 2 keywords.